# TERRAPYGE: Slope Unit Extraction (r.slopeunits)

This notebook extracts geomorphologically accurate slope units from the
conditioned DEM using the `r.slopeunits` algorithm via GRASS GIS.

**Prerequisites:**
- GRASS GIS 8.x+ installed and on PATH
- r.slopeunits add-on: `grass --tmp-mapset --exec g.extension extension=r.slopeunits`
- Conditioned DEM at `data/processed/CebuCity_DEM_conditioned_utm.tif`

**Parameters (from config.yaml):**
- Flow accumulation threshold: 200,000 sqm
- Reduction factor: 2
- Minimum area: 1,000,000 sqm
- Minimum circular variance: 0.1

## 1. Check Prerequisites

In [ ]:
import subprocess
import sys
from pathlib import Path
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import torch
from shapely.geometry import mapping

# Check GRASS
try:
    result = subprocess.run(['grass', '--version'], capture_output=True, text=True, timeout=30)
    print(f"GRASS GIS: {result.stdout.strip()}")
except FileNotFoundError:
    print("ERROR: GRASS GIS not found on PATH.")
    print("Install from: https://grass.osgeo.org/download/windows/")
    sys.exit(1)

# Check r.slopeunits
result = subprocess.run(
    ['grass', '--tmp-mapset', '--exec', 'r.slopeunits', '--help'],
    capture_output=True, text=True, timeout=30
)
if 'r.slopeunits' in result.stdout.lower() or 'r.slopeunits' in result.stderr.lower():
    print("r.slopeunits: available")
else:
    print("r.slopeunits NOT found. Install with:")
    print("  grass --tmp-mapset --exec g.extension extension=r.slopeunits")
    sys.exit(1)

## 2. Run r.slopeunits Extraction

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

# Paths
dem_path = Path.cwd() / 'data' / 'processed' / 'CebuCity_DEM_conditioned_utm.tif'
output_dir = Path.cwd() / 'data' / 'processed'
output_shp = output_dir / 'slope_units.shp'
output_geojson = output_dir / 'slope_units.geojson'

# r.slopeunits parameters (from config.yaml)
T = 200000  # flow accumulation threshold (sqm)
R = 2       # reduction factor
A = 1000000 # minimum area (sqm)
C = 0.1     # minimum circular variance

print(f"Input DEM: {dem_path}")
print(f"Parameters: t={T}, r={R}, a={A}, c={C}")

In [ ]:
# Find GRASS GIS (check common Windows install locations)
grass_cmd = 'grass'
try:
    subprocess.run([grass_cmd, '--version'], capture_output=True, timeout=10)
except FileNotFoundError:
    for candidate in [
        r'C:\OSGeo4W\bin\grass84.bat',
        r'C:\OSGeo4W64\bin\grass84.bat',
        r'C:\Program Files\GRASS GIS 8.4\grass84.bat',
    ]:
        if Path(candidate).exists():
            grass_cmd = candidate
            break
    else:
        print('ERROR: GRASS GIS not found. Install from https://grass.osgeo.org/download/windows/')
        sys.exit(1)

# Verify GRASS and r.slopeunits
result = subprocess.run([grass_cmd, '--version'], capture_output=True, text=True, timeout=30)
print(f'GRASS GIS: {result.stdout.strip()}')

# Write a Python script that runs all GRASS commands in one session
grass_py = output_dir / '_grass_slopeunits_run.py'
grass_py.write_text(f'''
import os
os.environ["GRASS_ADDON_BASE"] = os.path.join(os.environ.get("APPDATA", ""), "GRASS8", "addons")
import grass.script as gs

dem = r"{dem_path}"
out_shp = r"{output_shp}"
out_geo = r"{output_geojson}"

print("Importing DEM...")
gs.run_command("r.in.gdal", input=dem, output="dem", overwrite=True)
gs.run_command("g.region", raster="dem")

print("Running r.slopeunits.create...")
gs.run_command("r.slopeunits.create",
    demmap="dem", slumap="slope_units", slumapvect="slope_units_vec",
    thresh={T}, rf={R}, areamin={A}, cvmin={C},
    maxiteration=100, overwrite=True)

print("Exporting shapefile...")
gs.run_command("v.out.ogr", input="slope_units_vec", type="area",
    output=out_shp, format="ESRI_Shapefile", overwrite=True)
print("Exporting GeoJSON...")
gs.run_command("v.out.ogr", input="slope_units_vec", type="area",
    output=out_geo, format="GeoJSON", overwrite=True)
print("DONE")
''', encoding='utf-8')

# Create GRASS project with matching CRS
gisdbase = output_dir.parent / 'grassdb'
project = gisdbase / 'cebu_utm'

print('Creating GRASS project with matching CRS...')
result = subprocess.run(
    [grass_cmd, '-c', 'EPSG:32651', str(project),
     '--exec', 'python', str(grass_py)],
    capture_output=True, text=True, timeout=600
)

grass_py.unlink(missing_ok=True)

if result.returncode == 0:
    print('r.slopeunits completed successfully.')
else:
    print(f'ERROR (exit code {result.returncode}):')
    print(result.stderr)
    sys.exit(1)

## 3. Load and Validate Slope Units

In [ ]:
# Load slope units
gdf = gpd.read_file(output_shp)

print(f"Number of slope units: {len(gdf)}")
print(f"CRS: {gdf.crs}")
print(f"Columns: {list(gdf.columns)}")

# Area statistics (in sqm)
areas = gdf.geometry.area
print(f"\nArea statistics (sqm):")
print(f"  Min:    {areas.min():.0f}")
print(f"  Max:    {areas.max():.0f}")
print(f"  Mean:   {areas.mean():.0f}")
print(f"  Median: {areas.median():.0f}")
print(f"  Total:  {areas.sum():.0f}")

# Validate minimum area constraint
n_below_min = (areas < A).sum()
print(f"\nSlope units below min_area ({A} sqm): {n_below_min}")

# Check for empty/invalid geometries
n_empty = gdf.geometry.is_empty.sum()
print(f"Empty geometries: {n_empty}")

## 4. Visualize Slope Units

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Slope units map
gdf.plot(ax=axes[0], edgecolor='black', linewidth=0.3, color='lightblue', alpha=0.7)
axes[0].set_title(f'Slope Units (n={len(gdf)})', fontsize=14)
axes[0].set_xlabel('Easting (m)')
axes[0].set_ylabel('Northing (m)')

# Area distribution
areas_ha = areas / 10000  # convert to hectares
axes[1].hist(areas_ha, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(A / 10000, color='red', linestyle='--', label=f'Min area ({A/10000:.0f} ha)')
axes[1].set_xlabel('Area (hectares)')
axes[1].set_ylabel('Count')
axes[1].set_title('Slope Unit Area Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig(str(Path.cwd() / 'results' / 'figures' / 'slope_units_overview.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Figure saved to: results/figures/slope_units_overview.png")

## 5. Save Results

In [ ]:
import torch
from shapely.geometry import mapping

# Save as PyTorch-compatible metadata
su_metadata = {
    'n_units': len(gdf),
    'crs': str(gdf.crs),
    'total_area_sqm': float(areas.sum()),
    'mean_area_sqm': float(areas.mean()),
    'params': {'threshold': T, 'reduction': R, 'min_area': A, 'circular_variance': C},
}

print("Slope unit metadata:")
for k, v in su_metadata.items():
    print(f"  {k}: {v}")

print(f"\nOutputs:")
print(f"  Shapefile: {output_shp}")
print(f"  GeoJSON:   {output_geojson}")
print(f"\nSlope unit extraction complete.")

## 6. Next Steps

1. **Notebook 04: Graph Construction**
   - Load slope units shapefile.
   - Aggregate DEM features (elevation, slope, aspect, curvature, TWI, SPI) to slope units.
   - Build spatial edges (Queen/Rook contiguity via libpysal).
   - Build hydrological edges (D8 flow routing via pysheds).
   - Create PyTorch Geometric HeteroData graph.